# Agentic AI Assignment 3
**Name:** Laxmi Bhargav Mende\
**Roll No:** 25B0940

---
---

### Question 1: Build a ReAct Agent with LangChain
Tools used: `DuckDuckGoSearchRun`,a custom `Calculator` and a custon `currency convertor`

In [ ]:
import os
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.2,
    thinking_level="medium"
)

In [ ]:
import ast
import operator
import math

@tool
def calculator_tool(expression: str) -> str:
    """
    Evaluates a mathematical expression including scientific functions.
    Supported operations: +, -, *, /, **, sin, cos, tan, asin, acos, atan, log, log10, sqrt, exp, pi, e.
    """
    allowed_operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
    }

    allowed_functions = {
        'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
        'asin': math.asin, 'acos': math.acos, 'atan': math.atan,
        'log': math.log, 'log10': math.log10, 'sqrt': math.sqrt,
        'exp': math.exp
    }
    
    allowed_constants = {
        'pi': math.pi,
        'e': math.e
    }

    def _evaluate(node):
        # Modern Python (3.8+) uses ast.Constant for all numbers
        if isinstance(node, ast.Constant): 
            if isinstance(node.value, (int, float)):
                return node.value
            raise ValueError("Only numeric constants are allowed.")
            
        elif isinstance(node, ast.Name):
            if node.id in allowed_constants:
                return allowed_constants[node.id]
            raise ValueError(f"Unsupported variable or constant: {node.id}")
            
        elif isinstance(node, ast.BinOp):
            left = _evaluate(node.left)
            right = _evaluate(node.right)
            op = type(node.op)
            if op in allowed_operators:
                return allowed_operators[op](left, right)
            raise ValueError(f"Unsupported operator: {op}")
            
        elif isinstance(node, ast.UnaryOp):
            operand = _evaluate(node.operand)
            if isinstance(node.op, ast.USub):
                return -operand
            elif isinstance(node.op, ast.UAdd):
                return operand
            raise ValueError(f"Unsupported unary operator: {type(node.op)}")
            
        elif isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id in allowed_functions:
                args = [_evaluate(arg) for arg in node.args]
                return allowed_functions[node.func.id](*args)
            raise ValueError("Unsupported function call.")
            
        raise ValueError("Invalid expression format.")

    try:
        tree = ast.parse(expression, mode='eval').body
        result = _evaluate(tree)
        return str(result)
    except Exception as e:
        return f"Calculation Error: {e}"

In [ ]:
import requests

@tool
def currency_tool(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Converts an amount from one currency to another using the Frankfurter API.
    
    Supported Currencies (31): 
    AUD, BGN, BRL, CAD, CHF, CNY, CZK, DKK, EUR, GBP, HKD, HUF, IDR, ILS, INR, 
    ISK, JPY, KRW, MXN, MYR, NOK, NZD, PHP, PLN, RON, SEK, SGD, THB, TRY, USD, ZAR
    
    Example: currency_tool(100.0, "USD", "EUR")
    """
    from_curr = from_currency.upper().strip()
    to_curr = to_currency.upper().strip()
    
    if from_curr == to_curr:
        return f"{amount} {from_curr} = {amount} {to_curr}"

    try:
        url = "https://api.frankfurter.app/latest"
        params = {
            "amount": amount,
            "from": from_curr,
            "to": to_curr
        }
        
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        converted_amount = data['rates'][to_curr]
        return f"{amount} {from_curr} = {converted_amount} {to_curr}"
        
    except requests.exceptions.RequestException as e:
        return f"Network or API Error: {e}"
    except KeyError:
        return f"Conversion Error: Ensure currency codes are from the supported list."

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun(
    description="""A web search engine tool that retrieves real-time internet search results and web snippets using DuckDuckGo.
    Use this tool when you need up-to-date facts, current events, or external information not present in your training data.
    The input must be a specific, concise search query string."""
)

In [ ]:
#testing the tools once
print(calculator_tool.invoke("(sin(pi/2)*log10(100))*(44 + (1 ** 23))"))

print(currency_tool.invoke({
    "amount" : 200,
    "from_currency" : "USD",
    "to_currency" : "INR"
    }))

print(search_tool.invoke("When was the most recent Cricket World Cup and Which team won it?"))

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model = llm,
    tools = [calculator_tool, search_tool, currency_tool],
    debug = False                 # set to False to stop the live updates about the agents current state during invoke or batch
)
agent

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

questions = [
    {"messages" : [HumanMessage("What is the official launch price of the Apple iPhone17 Pro Max in USD, and what is that exact amount converted into Rupees (INR)?")]},
    {"messages" : [HumanMessage("Find the exact distance between the Earth and the Moon in kilometers. If a spacecraft travels at a constant speed of 5,500 km/h, how many hours will the journey take?")]},
    {"messages" : [HumanMessage("Find the current price of a single share of Tesla (TSLA) stock in USD. If I want to purchase 45 shares, calculate the total cost and then convert that final total into Indian Rupees (INR).")]}
]

results = agent.batch(questions)

In [ ]:
for i in range(0, len(questions)):
    print(questions[i]["messages"][0].content)
    print("Ans: ",results[i]['messages'][-1].content[0]['text'])
    print("\n",'~'*70,"\n")


In [ ]:
#The internal process of the Agent for one question

results[0]['messages']

In [ ]:
# As we can see above:
# 1.Human Message : Our Prompt
# 2.Understands to search for phone price : Uses Search Engine
# 3.Tools Response: Price in USD is missing
# 4.Resends query to search engine to get price
# 5.Tool Response : With an offered price
# 6.Function call to currency convertor
# 7.CurrencyConvertor's response
# 8.Final Output

---
---
### Question 2: Build a LangGraph Workflow with Cycles

In [ ]:
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

generativellm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=1,
    thinking_level="high",
)

class Evaluation(BaseModel):
    evaluation_result: bool = Field(..., description="True if the answer is accurate, complete, and well-supported by sources. False otherwise.")
    explanation: str = Field(..., description="A brief explanation of the shortcomings if the answer is not satisfactory. If the answer is satisfactory, this field should be empty.")
    rating: int = Field(..., description="""A rating from 0 to 100 indicating the quality of the answer, with 100 being the best possible answer.
            This rating should be based on the quality of the answer, the accuracy of the information provided, and the clarity of the explanation.
            This needs to be independent of the evaluation_result and explanation fields, and should be a numerical value between 0 and 100.""")

evaluation_llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
)
evaluation_llm = evaluation_llm.with_structured_output(Evaluation)

In [ ]:
import re, operator
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, SystemMessage

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    drafts: Annotated[list[str], operator.add]
    answer : Annotated[str, "The final answer generated by the agent or the Review of the answer."]
    iteration_count: Annotated[int, "The number of iterations the agent has gone through."]
    passed_evaluation: Annotated[bool, "Indicates whether the generated answer passed evaluation criteria."]

def generate_node(state: AgentState):
    messages = state.get("messages", [])
    print(f"Generating Draft {state.get('iteration_count', 0) + 1}")
    response = generativellm.invoke(messages)
    return {"messages": [response], "drafts": [response.content], "answer": response.content, "iteration_count": state.get("iteration_count", 0) + 1}

def evaluate_node(state: AgentState):
    latest_message = state['answer']
    question = state['messages'][0].content
    print(f"Evaluating Draft {state.get('iteration_count', 0)}")
    evaluation = evaluation_llm.invoke([
        SystemMessage(content="You are an evaluator that assesses the quality of answers provided by an AI assistant. Your task is to determine whether the answer is accurate, complete, and well-supported by sources and is of atleast 50 words. If the answer meets these criteria, respond with 'True'. If it does not, respond with 'False' and provide a brief explanation of the shortcomings."),
        HumanMessage(content=f"Question: {question}\nAnswer: {latest_message}\nPlease evaluate the answer based on accuracy, completeness, and citation of sources. Provide a boolean evaluation_result, an explanation if the answer is unsatisfactory, and a rating from 0 to 100.")
    ])
    ans = f"Rating: {evaluation.rating}/100\nExplanation: {evaluation.explanation}"
    return {"passed_evaluation": evaluation.evaluation_result, "answer": ans}

def refine_node(state: AgentState):
    print(f"Refining Prompt for next draft")
    refinement_prompt = HumanMessage(
        content=f"The previous answer was not satisfactory.\nReview: {state['answer']}. Please improve the answer by addressing the shortcomings mentioned in the review, ensuring that it is accurate, complete, well-supported by sources, and is of atleast 50 words.")
    return {"messages": [refinement_prompt]}

def pass_evaluation(state: AgentState) -> bool:
    match = re.search(r"Rating: (\d+)/100", state['answer'])
    rating = int(match.group(1)) if match else 0
    if ( state['iteration_count'] > 3 or ( state["passed_evaluation"] and rating >= 75 )) and state['iteration_count'] > 1: # Loops atleast 1 and atmost thrice
        print(f"Draft {state['iteration_count']} evaluation passed with rating {rating}/100.")
        return True
    else:
        print(f"Draft {state['iteration_count']} evaluation failed with rating {rating}/100.")
        return False


In [ ]:
workflow = StateGraph(AgentState)
workflow.add_node("generate_node", generate_node)
workflow.add_node("evaluate_node", evaluate_node)
workflow.add_node("refine_node", refine_node)

workflow.add_edge(START, "generate_node")
workflow.add_edge("generate_node", "evaluate_node")
workflow.add_conditional_edges("evaluate_node", pass_evaluation, {True: END, False: "refine_node"})
workflow.add_edge("refine_node", "generate_node")

app = workflow.compile(
    debug=False  # Set to True to stop the live values about the agents current state during invoke or batch or stream
)


In [ ]:
from IPython.display import Image, display

# Generate the graph and display it as a PNG
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
question = "Explain the concept of Agentic AI."
initial_state = {
    "messages": [HumanMessage(content=question)],
    'drafts': [],
    "iteration_count": 0,
    "passed_evaluation": False
}

response = app.invoke(initial_state)

In [ ]:
print("FINAL STATE:")
print("Messages:",response['messages'])
print("Drafts:",response['drafts'])


---
---

### Question 3: Cold Email Agent via Gmail MCP

In [ ]:
import logging
logging.disable(logging.WARNING)

In [ ]:
from pydantic import BaseModel
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI

class EmailState(TypedDict):
    recipient_email: str
    recipient_name: str
    context: str
    draft: str
    passed_review: bool
    comments: str
    messages: Annotated[list, add_messages]

class EvaluationOutput(BaseModel):
    evaluation_result: bool
    explanation: str
    rating: int

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.2,
    thinking_level="medium"
)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

async def run_email_agent():
    print("Connecting to MCP Client...")
    client = MultiServerMCPClient({
        "gmail_server": {
            "command": "npx",
            "args": ["-y", "@gongrzhe/server-gmail-autoauth-mcp"],
            "transport": "stdio"
        }
    })
    global gmail_mcp_tools
    gmail_mcp_tools = await client.get_tools()
    print("MCP Client Connected.")

await run_email_agent()

In [ ]:
from langchain.agents import create_agent

gmail_agent = create_agent(
    model=llm,
    tools=gmail_mcp_tools,
)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END


def draft_node(state: EmailState):
    print("Beginning Email Draft")
    prompt = HumanMessage(content=f"Draft a professional cold email to {state['recipient_name']} regarding {state['context']}. Include a clear Subject line.")
    response = llm.invoke([prompt])
    return {"draft": response.text, "messages": [prompt,response]}

def review_node(state: EmailState):
    print("Reviewing Draft")
    review_llm = llm.with_structured_output(EvaluationOutput)
    updates = [
        SystemMessage(content="You are an evaluator that assesses the quality of email drafts. Your task is to determine whether the draft is professional, clear, and well-structured. If the draft meets these criteria, respond with 'True'. If it does not, respond with 'False' and provide a brief explanation of the shortcomings."),
        HumanMessage(content=f"Draft: {state['draft']}\nPlease evaluate the draft based on professionalism, clarity, and structure. Provide a boolean evaluation_result and an explanation if the draft is unsatisfactory and also rate it on a scale of 10.")
    ]
    evaluation = review_llm.invoke(updates)
    return {"passed_review": evaluation.evaluation_result, 
            "comments": f"{evaluation.explanation} (Rating: {evaluation.rating}/10)", 
            'messages': updates+[AIMessage(content=f"Evaluation Result: {evaluation.evaluation_result}, Explanation: {evaluation.explanation}, Rating: {evaluation.rating}/10")]}

async def send_node(state: EmailState):
    print("Sending the Email via Gmail MCP..")
    prompt = HumanMessage(content=f"Use the available gmail tool to send the following email to {state['recipient_email']}. \n\n{state['draft']}")
    response = await gmail_agent.ainvoke({'messages': [prompt]})
    print(response['messages'])
    return {"comments": "Email Sent Successfully",
            "messages": [prompt, response['messages'][0]]}

def pass_evaluation(state: EmailState) -> bool:
    if state["passed_review"]:
        print("Draft passed review and is ready to be sent.")
        return True
    print("Draft Review Failed. Returning to draft stage.")
    return False

email_workflow = StateGraph(EmailState)
email_workflow.add_node("draft_node", draft_node)
email_workflow.add_node("review_node", review_node)
email_workflow.add_node("send_node", send_node)

email_workflow.add_edge(START, "draft_node")
email_workflow.add_edge("draft_node", "review_node")
email_workflow.add_conditional_edges("review_node", pass_evaluation, {True: "send_node", False: "draft_node"})
email_workflow.add_edge("send_node", END)

email_app = email_workflow.compile()

In [ ]:
from IPython.display import Image, display

# Generate the graph and display it as a PNG
display(Image(email_app.get_graph().draw_mermaid_png()))

In [ ]:

inputs = {
    "recipient_email": "25b0940@iitb.ac.in", 
    "recipient_name": "Dr. Smith",
    "context": "Have been following his research and want to join his lab for a summer internship on Multi-Agent Systems.",
    "draft": "",
    "passed_review": False,
    "messages": []
}


In [ ]:
final_email_state = await email_app.ainvoke(inputs)

print("COMPLETED")

In [ ]:

print("Final Email State:")
print(final_email_state)